# Geometry 02b — Saturation et Nullstellensatz : l'opération derrière l'astuce

**Public : Licence** — accrétion du notebook [02](Geometry-02-From-Equation-To-Proof.ipynb) (*From Equation to Proof*), qu'elle suppose lu ; la série se lit sans ouvrir cette lettre (cf [README](README.md)).


## Où en sommes-nous ?

Le notebook 02 s'est arrêté sur un geste qui a l'air d'un tour de passe-passe : pour sauver le théorème « *B orthogonal à C et à D implique C, D alignés avec A* », il a adjoint l'équation $z\cdot g - 1 = 0$ avec $g = x_B^2 + y_B^2$ — rendre $g$ **inversible** — et la conclusion s'est réduite à zéro dans l'idéal élargi. Le notebook 03, de son côté, vérifie ses preuves de Wu par un test voisin : $\langle H,\ z J - 1,\ w g - 1\rangle = \langle 1\rangle$, l'« astuce de Rabinowitsch ».

Ces deux gestes manipulent le **même objet sans jamais le nommer** : la **saturation** $I : g^\infty$ — l'idéal des conclusions valables sur le lieu non dégénéré $V(I)\setminus V(g)$. Ce carnet le nomme, le **calcule comme un idéal du plan** (pas seulement comme un test d'appartenance), dérive le **Nullstellensatz fort** depuis le faible par la même astuce, et écrit enfin le contrat exact que la série signe quand elle dit « démontre modulo les dégénérescences ».

Fil rouge inchangé : le théorème du milieu de l'hypoténuse, dans son encodage à quatre points du 02 §6.


## Plan

1. §1 — Le fil rouge, troisième visite : objets, rappel du geste du 02 §7.
2. §2 — Colon et saturation $I : g^n$, $I : g^\infty$ : la chaîne calculée, puis l'idéal saturé **exhibé** par élimination.
3. §3 — L'astuce de Ritt–Rabinowitsch relue : $\langle I, zg-1\rangle \cap k[x] = I : g^\infty$ — l'astuce EST l'algorithme.
4. §4 — Nullstellensatz : du faible au fort, la dérivation classique **exécutée** avec cofacteurs sympy.
5. §5 — Le témoin négatif qui **survit** à la saturation (rectangle non isocèle) — l'organe Z3 trouve un contre-exemple non dégénéré.
6. §6 — Le prix : trois recettes, deux moteurs, mesures.
7. §7 — Le contrat de la série relu : chaque critère du 02, du 03 (Wu, Rabinowitsch) est un énoncé de saturation.
8. §8–§10 — Trois exercices.


In [1]:
import time
import sympy as sp

t_start = time.perf_counter()
print("sympy", sp.__version__)


sympy 1.14.0


### §1 — Le fil rouge, troisième visite

On reconstruit l'encodage du 02 §6 : $A$ à l'origine, les points $B, C, D$ libres, deux hypothèses d'orthogonalité, une conclusion d'alignement. Le théorème est **faux** tel quel (témoin $B = A$ du 02 §6, où les deux hypothèses sont trivialement nulles) et le 02 §7 l'a sauvé en inversant $g = x_B^2 + y_B^2$ dans un anneau élargi.


In [2]:
# Les objets du fil rouge, comme au 02 §6
xB, yB, xC, yC, xD, yD = sp.symbols('x_B y_B x_C y_C x_D y_D')
PLAN = (xB, yB, xC, yC, xD, yD)          # les six inconnues du plan

I_gens = [xB * xC + yB * yC,             # H2a : AB scalaire AC = 0
          xB * xD + yB * yD]             # H2b : AB scalaire AD = 0
CONCL = xC * yD - xD * yC                # A, C, D alignes (produit vectoriel)
g_nd = xB ** 2 + yB ** 2                 # non-degenerescence B != A

print("I  = <", I_gens, ">")
print("CONCL =", CONCL, "| g_nd =", g_nd)


I  = < [x_B*x_C + y_B*y_C, x_B*x_D + y_B*y_D] >
CONCL = x_C*y_D - x_D*y_C | g_nd = x_B**2 + y_B**2


In [3]:
# Rappel du geste du 02 §7 : inverser g dans un anneau elargi
z = sp.symbols('z')
G_elargi = sp.groebner(I_gens + [z * g_nd - 1], *PLAN, z, order='lex')
reste_02 = G_elargi.reduce(CONCL)[1]
print("reste de CONCL dans <I, z*g-1> :", reste_02, "| nul ?", reste_02 == 0)
print("(c'est le verdict du 02 §7 : le theoreme tient sous B != A)")


reste de CONCL dans <I, z*g-1> : 0 | nul ? True
(c'est le verdict du 02 §7 : le theoreme tient sous B != A)


#### Lecture du résultat

Ce reste nul dit une chose précise mais détournée : $\mathrm{CONCL} \in \langle I,\ zg-1\rangle$, un idéal de l'anneau **à sept variables** $k[x_B,\dots,y_D, z]$. La variable $z$ n'a aucun sens géométrique — elle est là pour inverser $g$. Trois questions restent sans réponse :

- quel **idéal du plan** (six variables, sans $z$) ce reste nul atteste-t-il ?
- qu'est-ce que cette appartenance **garantit géométriquement** (sur quels points, dans quel corps) ?
- quel est le lien exact avec le test $\langle H, zJ-1, wg-1\rangle = \langle 1\rangle$ du 03 §7 ?

Les trois réponses portent le même nom : la saturation.


### §2 — Le colon et la saturation : $I : g^\infty$

**Définition.** Pour un idéal $I$ et un polynôme $g$ :

$$I : g^n = \{\, f \ :\ f\, g^n \in I \,\}, \qquad I : g^\infty = \bigcup_{n \geq 1} I : g^n.$$

La chaîne $I : g \subseteq I : g^2 \subseteq \cdots$ croit et **stabilise** (l'anneau est noethérien) : l'union est un idéal. Lecture géométrique : $I : g^\infty$ est le plus grand idéal qui **coincide avec $I$ partout où $g \neq 0$** — les conclusions valables sur le lieu non dégénéré $V(I) \setminus V(g)$.

Première visite : à la main, par la chaîne. On cherche le plus petit $n$ tel que $\mathrm{CONCL}\cdot g^n \in I$ — c'est exactement ce que la pseudo-division du 03 établit pour ses propres systèmes.


In [4]:
# La chaine a la main : plus petit n tel que CONCL * g^n appartienne a I
G6 = sp.groebner(I_gens, *PLAN, order='lex')
for n in range(1, 4):
    restes = G6.reduce(sp.expand(CONCL * g_nd ** n))
    print(f"n = {n} : reste de CONCL*g^{n} par I : {restes[1]}  | nul ? {restes[1] == 0}")


n = 1 : reste de CONCL*g^1 par I : 0  | nul ? True
n = 2 : reste de CONCL*g^2 par I : 0  | nul ? True
n = 3 : reste de CONCL*g^3 par I : 0  | nul ? True


#### Lecture du résultat

Le reste s'annule dès $n = 1$ : $\mathrm{CONCL}\cdot g \in I$, donc $\mathrm{CONCL} \in I : g \subseteq I : g^\infty$. Ce n'est pas un miracle — les cofacteurs sont explicites, et la machine peut les exhiber :


In [5]:
# Les cofacteurs explicites de l'appartenance CONCL*g in I
# deux identites de base, chacune verifiee par developpement
id1 = sp.expand(xB * CONCL - (yD * I_gens[0] - yC * I_gens[1]))
id2 = sp.expand(yB * CONCL - (xC * I_gens[1] - xD * I_gens[0]))
print("xB*CONCL = yD*H2a - yC*H2b  -> ecart :", id1, "| nul ?", id1 == 0)
print("yB*CONCL = xC*H2b - xD*H2a  -> ecart :", id2, "| nul ?", id2 == 0)
# en les combinant : g*CONCL = xB*(xB*CONCL) + yB*(yB*CONCL)
q1, q2 = xB * yD - yB * xD, yB * xC - xB * yC
ecart = sp.expand(CONCL * g_nd - (q1 * I_gens[0] + q2 * I_gens[1]))
print("cofacteurs : q1 =", q1, "| q2 =", q2)
print("CONCL*g - (q1*H2a + q2*H2b) =", ecart, "| nul ?", ecart == 0)


xB*CONCL = yD*H2a - yC*H2b  -> ecart : 0 | nul ? True
yB*CONCL = xC*H2b - xD*H2a  -> ecart : 0 | nul ? True
cofacteurs : q1 = x_B*y_D - x_D*y_B | q2 = -x_B*y_C + x_C*y_B
CONCL*g - (q1*H2a + q2*H2b) = 0 | nul ? True


#### Lecture du résultat

La chaîne a répondu **pour cette conclusion-là**. Mais un idéal est plus qu'un test d'appartenance : on veut les **générateurs** de $I : g^\infty$, un objet du plan à six variables, sans variable auxiliaire. C'est le travail de l'élimination :

$$I : g^\infty \;=\; \langle I,\ 1 - t\cdot g \rangle \ \cap\ k[x_B, \dots, y_D]$$

avec $t$ une variable fraîche, éliminée par une base de Groebner lexicographique (théorème d'élimination : les éléments de la base qui ne contiennent pas $t$ engendrent l'intersection).


In [6]:
# L'ideal sature exhibe par elimination : <I, 1 - t*g> intersecte k[PLAN]
t = sp.symbols('t')
G_sat = sp.groebner(I_gens + [t * g_nd - 1], *PLAN, t, order='lex')
sat_gens = [p.as_expr() if hasattr(p, 'as_expr') else p for p in G_sat.polys
            if not (p.as_expr().has(t) if hasattr(p, 'as_expr') else p.has(t))]
sat_gens = [sp.expand(p) for p in sat_gens]
print("generateurs de I : g^infini (sans t) :")
for p in sat_gens:
    print("   ", p)


generateurs de I : g^infini (sans t) :
    x_B*x_C + y_B*y_C
    x_B*x_D + y_B*y_D
    x_C*y_D - x_D*y_C


In [7]:
# Le theoreme, redevenu un enonce du plan : CONCL appartient-il a l'ideal sature ?
G_plane = sp.groebner(sat_gens, *PLAN, order='lex')
reste_sat = G_plane.reduce(CONCL)[1]
print("reste de CONCL par I : g^infini :", reste_sat, "| nul ?", reste_sat == 0)
print("generation du plan :", len(sat_gens), "polynomes, zero variable auxiliaire")


reste de CONCL par I : g^infini : 0 | nul ? True
generation du plan : 3 polynomes, zero variable auxiliaire


#### Lecture du résultat

Voilà l'objet que les deux carnets précédents manipulaient sans le montrer : un idéal **du plan**, engendré par les hypothèses et (sous une forme ou une autre) la conclusion elle-même. Le théorème s'énonce maintenant sans variable artificielle :

$$\mathrm{CONCL} \in I : g_{nd}^\infty \quad\text{— « l'alignement est valable partout où } B \neq A \text{ et les hypothèses tiennent »}.$$

Vérification de cohérence : la chaîne et l'élimination doivent raconter la même histoire. On calcule $I : g^n$ pour $n = 1, 2, 3$ (même recette d'élimination, avec $g^n$) et on regarde la chaîne se stabiliser **sur** l'idéal saturé.


In [8]:
# Cross-validation : la chaine I:g^n stabilise sur l'ideal sature
def saturation_par_elimination(gens_ideal, g, variables, puissance=1):
    """Rend la liste des generateurs de <I, 1 - t*g^p> intersecte k[variables]."""
    u = sp.symbols('_u')
    G = sp.groebner(gens_ideal + [u * g ** puissance - 1], *variables, u, order='lex')
    return [sp.expand(p.as_expr()) for p in G.polys if not p.as_expr().has(u)]

chain = {}
for n in (1, 2, 3):
    chain[n] = saturation_par_elimination(I_gens, g_nd, PLAN, puissance=n)
    gn = sp.groebner(chain[n], *PLAN, order='lex')
    contient_concl = gn.reduce(CONCL)[1] == 0
    print(f"I : g^{n} : {len(chain[n])} generateurs | contient CONCL : {contient_concl}")

# stabilisation : chaque etage doit etre contenu dans le suivant, et le dernier = sature
for n in (1, 2):
    Gn, Gnext = sp.groebner(chain[n], *PLAN, order='lex'), sp.groebner(chain[n + 1], *PLAN, order='lex')
    incl = all(Gnext.reduce(p)[1] == 0 for p in chain[n])
    print(f"I : g^{n}  inclus dans I : g^{n+1} : {incl}")
G3_vs_sat = sp.groebner(chain[3], *PLAN, order='lex')
meme = (all(G3_vs_sat.reduce(p)[1] == 0 for p in sat_gens)
        and all(G_plane.reduce(p)[1] == 0 for p in chain[3]))
print("I : g^3 = I : g^infini (mutuellement) :", meme)


I : g^1 : 3 generateurs | contient CONCL : True
I : g^2 : 3 generateurs | contient CONCL : True


I : g^3 : 3 generateurs | contient CONCL : True
I : g^1  inclus dans I : g^2 : True
I : g^2  inclus dans I : g^3 : True


I : g^3 = I : g^infini (mutuellement) : True


#### Lecture du résultat

Trois façons de dire la même chose, maintenant toutes **exécutées** : $\mathrm{CONCL}\cdot g \in I$ (la chaîne), $\mathrm{CONCL} \in I:g^\infty$ (l'appartenance), et les générateurs de l'idéal saturé (l'objet). Il reste à comprendre pourquoi l'astuce du 02 — ajouter $zg - 1$ — calcule *exactement* cet idéal.


### §3 — L'astuce de Ritt–Rabinowitsch, relue comme un algorithme

**Théorème (Ritt–Rabinowitsch).** $\quad \langle I,\ zg - 1\rangle \cap k[x] \;=\; I : g^\infty$.

*Esquisse.* L'inclusion $\supseteq$ : si $f g^n = \sum a_i h_i$, multiplier par $z^n$ et retrancher $z^n g^n - 1 = (zg-1)(\dots)$ fait de $f$ une combinaison de $I$ et de $zg-1$, sans $z$ dans le résultat. L'inclusion $\subseteq$ : une identité $f = \sum a_i h_i + b(zg - 1)$, substituée en $z = 1/g$ puis multipliée par $g^m$ ($m$ assez grand), donne $f g^m \in I$. La vérification machine tient dans deux reduce :


In [9]:
# Verification du theoreme sur le fil rouge, dans les deux sens
# sens 1 : les generateurs de l'ideal sature vivent dans <I, z*g-1> intersecte k[PLAN]
dans_elargi = all(G_elargi.reduce(p)[1] == 0 for p in sat_gens)
print("sat(I,g) inclus dans <I, z*g-1> cap k[PLAN] :", dans_elargi)

# sens 2 : CONCL*g (membre de I, donc de l'intersection) redonne le verdict du 02
identite_02 = G_elargi.reduce(CONCL)[1] == 0
print("CONCL dans <I, z*g-1> (rappel 02 §7) :", identite_02,
      "| CONCL dans I:g^infini (02b §2) :", reste_sat == 0)
print("deux routes, un seul verdict :", identite_02 == (reste_sat == 0))


sat(I,g) inclus dans <I, z*g-1> cap k[PLAN] : True
CONCL dans <I, z*g-1> (rappel 02 §7) : True | CONCL dans I:g^infini (02b §2) : True
deux routes, un seul verdict : True


#### Lecture du résultat — les trois critères de la série, un seul objet

| Carnet | Critère posé | Énoncé de saturation équivalent | Force |
|---|---|---|---|
| 02 §7 | reste nul dans $\langle I, zg-1\rangle$ | $f \in I : g^\infty$ | forte |
| 03 §5 (Wu) | pseudo-reste nul dans la chaîne | $f\, J^k \in I$, donc $f \in I : J^\infty$ | forte |
| 03 §7 (Rabinowitsch) | $\langle H, zJ{-}1, wg{-}1\rangle = \langle 1\rangle$ | $g \in \sqrt{I : J^\infty}$ | radiciel |

Les deux premiers critère prouvent une **appartenance à l'idéal saturé**, le troisième une appartenance à son **radical** — strictement plus faible en général ($f \in I:g^\infty \Rightarrow f^m \in I:g^\infty$... non : $f \in \sqrt{I:g^\infty}$ ne donne que $f^m \in I:g^\infty$). Le Nullstellensatz, section suivante, dit ce que le critère radiciel garantit quand même — et pourquoi il suffit pour la question géométrique.


### §4 — Nullstellensatz : du faible au fort, la dérivation exécutée

**Faible (Hilbert).** Sur $\mathbb{C}$ : un idéal propre $J \subsetneq k[x]$ a un zéro commun : $V(J) \neq \varnothing$.

Le faible répond « le système a-t-il une solution ? » mais pas « quel polynôme s'annule sur $V(J)$ ? ». La réponse est le **fort** :

$$I(V(J)) = \sqrt{J} \quad\text{— les polynômes nuls sur } V(J) \text{ sont exactement } \sqrt{J}.$$

La dérivation classique tient en trois lignes, et la machine peut la *jouer* sur un exemple. Prenons $J = \langle x^2\rangle$, $f = x$ : $f$ s'annule sur $V(J) = \{0\}$, donc le fort promet $x \in \sqrt{J}$, c'est-à-dire $x^2 \in J$. D'où vient cette puissance ?


In [10]:
# Etape 1 : le faible dit que <x^2, 1 - t*x> = <1> (aucun zero commun)
x, y_var = sp.symbols('x y')
J_toy = [x ** 2]
f_toy = x
G_ref = sp.groebner(J_toy + [1 - t * f_toy], x, t, order='lex')
print("base de <x^2, 1 - t*x> :", G_ref.polys if G_ref.polys else G_ref)
est_un = (len(G_ref.polys) == 1 and sp.expand(G_ref.polys[0].as_expr()) == 1)
print("l'ideal est <1> (faible NSZ) :", est_un)


base de <x^2, 1 - t*x> : [Poly(1, x, t, domain='ZZ')]
l'ideal est <1> (faible NSZ) : True


In [11]:
# Etape 2 : l'identite de Rabinowitsch, construite puis verifiee
b_cof = 1 + t * x              # degre 1 en t
a_cof = t ** 2                 # degre 2 en t
identite = sp.expand(sp.Integer(1) - (b_cof * (1 - t * x) + a_cof * x ** 2))
print("cofacteurs : b =", b_cof, "| a =", a_cof)
print("verification 1 - (b*(1-t*x) + a*x^2) =", identite, "| nul ?", identite == 0)

# la substitution t = 1/x, puis le nettoyage par x^m assez grand
m = max(sp.degree(b_cof, t), sp.degree(a_cof, t))
coeff_b = sp.expand(b_cof.subs(t, 1 / x) * x ** m)
coeff_a = sp.expand(a_cof.subs(t, 1 / x) * x ** m)
membre_droit = sp.expand(coeff_b * sp.Integer(0) + coeff_a * x ** 2)
print(f"apres substitution et nettoyage par x^{m} :")
print("   cofacteur polynomial devant x^2 :", coeff_a)
print("   x^2 =", membre_droit, "| certificat exact :",
      sp.expand(membre_droit - x ** 2) == 0)


cofacteurs : b = t*x + 1 | a = t**2
verification 1 - (b*(1-t*x) + a*x^2) = 0 | nul ? True
apres substitution et nettoyage par x^2 :
   cofacteur polynomial devant x^2 : 1
   x^2 = x**2 | certificat exact : True


In [12]:
# Etape 3 : le certificat obtenu, et la puissance minimale
G_J = sp.groebner(J_toy, x, order='lex')
p_min = next(p for p in range(1, 5) if G_J.reduce(f_toy ** p)[1] == 0)
print(f"la derivation livre f^{m} in J :", G_J.reduce(f_toy ** m)[1] == 0)
print(f"exposant minimal : {p_min} (x^{p_min - 1} in J :",
      G_J.reduce(f_toy ** (p_min - 1))[1] == 0, "— la puissance est necessaire)")


la derivation livre f^2 in J : True
exposant minimal : 2 (x^1 in J : False — la puissance est necessaire)


#### Lecture du résultat

Trois lignes de mathématiques, trois cellules de machine : le **faible** donne $1 \in \langle J, 1 - tf\rangle$ ; la **substitution** $t \mapsto 1/f$ suivie du nettoyage par $f^m$ transforme cette identité en certificat $f^m \in J$ ; c'est le **fort**. Aucune étape ne dépend de l'exemple — l'astuce est la même que celle du §3, appliquée au radical.

Pour la série, le fort a une conséquence de contrat : sur $\mathbb{C}$,

$$f \in \sqrt{I : g^\infty} \iff f \text{ s'annule sur } V(I) \setminus V(g).$$

Le critère radiciel du 03 §7 teste donc *exactement* la question géométrique « la conclusion s'annule-t-elle sur les figures non dégénérées ? » — et le critère fort du 02 (l'appartenance saturée) est **plus fort que nécessaire** pour cette question. Vérifions le contrat sur le fil rouge :


In [13]:
# Le contrat execute : CONCL s'annule sur V(I) prive de V(g) ?
w = sp.symbols('w')
G_contrat = sp.groebner(sat_gens + [1 - w * CONCL], *PLAN, w, order='lex')
est_un = (len(G_contrat.polys) == 1 and sp.expand(G_contrat.polys[0].as_expr()) == 1)
print("<I:g^infini, 1 - w*CONCL> = <1> :", est_un)
print("(vrai : aucun point non degenere ou les hypotheses tiennent et CONCL != 0)")


<I:g^infini, 1 - w*CONCL> = <1> : True
(vrai : aucun point non degenere ou les hypotheses tiennent et CONCL != 0)


### §5 — Le témoin négatif qui survit à la saturation

La saturation n'est pas une machine à prouver : elle **retire l'échappatoire dégénérée**, rien d'autre. Si un énoncé faux possède un contre-exemple **non dégénéré**, aucune saturation ne le sauvera. Retour au témoin du 02 §5 : « *un triangle rectangle est isocèle en A* » — faux, témoin 3-4-5 choisi à la main. Soumettons-le à toute la machinerie, puis demandons à l'organe réel ([`real_geometry_bridge`](real_geometry_bridge.py), Z3) de trouver lui-même un témoin et de dire s'il est dégénéré.


In [14]:
# Le faux theoreme du 02 §5, passe a la moulinette de la saturation
xA, yA, xB2, yB2, xC2, yC2 = sp.symbols('x_A y_A x_B2 y_B2 x_C2 y_C2')
PLAN3 = (xA, yA, xB2, yB2, xC2, yC2)
H_rect = (xB2 - xA) * (xC2 - xA) + (yB2 - yA) * (yC2 - yA)   # angle droit en A
Ciso = sp.expand((xB2 - xA) ** 2 + (yB2 - yA) ** 2
                 - ((xC2 - xA) ** 2 + (yC2 - yA) ** 2))       # AB = AC ?
g_rect = (xB2 - xA) ** 2 + (yB2 - yA) ** 2                    # B != A

sat_rect = saturation_par_elimination([H_rect], g_rect, PLAN3)
G_rect = sp.groebner(sat_rect, *PLAN3, order='lex')
reste_rect = G_rect.reduce(Ciso)[1]
print("reste de Ciso par sat(<H>, g) :", reste_rect, "| nul ?", reste_rect == 0)

w2 = sp.symbols('w2')
G_contre = sp.groebner(sat_rect + [1 - w2 * Ciso], *PLAN3, w2, order='lex')
est_un = (len(G_contre.polys) == 1 and sp.expand(G_contre.polys[0].as_expr()) == 1)
print("radicalement, Ciso s'annule-t-elle sur les figures non degenerees ?", est_un)


reste de Ciso par sat(<H>, g) : -2*x_A*x_B2 + 2*x_A*x_C2 + x_B2**2 - x_C2**2 - 2*y_A*y_B2 + 2*y_A*y_C2 + y_B2**2 - y_C2**2 | nul ? False
radicalement, Ciso s'annule-t-elle sur les figures non degenerees ? False


In [15]:
# L'organe reel : Z3, deux fois — sans puis avec la contrainte non degenere
import sys
from pathlib import Path
for _p in (Path.cwd(), Path.cwd() / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean" / "Geometry"):
    if (_p / "real_geometry_bridge.py").exists():
        sys.path.insert(0, str(_p))

import real_geometry_bridge as rbg
print("organe pret :", Path(rbg.__file__).name)

def decrire(verdict, etiquette):
    print(f"--- {etiquette}")
    print("   verdict :", verdict)
    if verdict.witness:
        print("   temoin :", {k.name: v for k, v in verdict.witness.items()})
        g_au = g_rect.subs(verdict.witness)
        print("   g au temoin (= |AB|^2) :", g_au, "| degenere (g = 0) ?", g_au == 0)

decrire(rbg.decide_real([H_rect], Ciso, list(PLAN3)),
        "sans contrainte : Z3 prend la porte degenere s'il peut")
g_CA = (xC2 - xA) ** 2 + (yC2 - yA) ** 2       # C != A
decrire(rbg.decide_real([H_rect], Ciso, list(PLAN3), assumptions=[g_rect, g_CA]),
        "avec les contraintes B != A et C != A")


organe pret : real_geometry_bridge.py


--- sans contrainte : Z3 prend la porte degenere s'il peut
   verdict : INVALID — temoin reel exact
   temoin : {'x_A': 0, 'y_A': 0, 'x_B2': 0, 'y_B2': 0, 'x_C2': 0, 'y_C2': -1}
   g au temoin (= |AB|^2) : 0 | degenere (g = 0) ? True
--- avec les contraintes B != A et C != A
   verdict : INVALID — temoin reel exact
   temoin : {'x_A': 0, 'y_A': 0, 'x_B2': -1, 'y_B2': 0, 'x_C2': 0, 'y_C2': -2}
   g au temoin (= |AB|^2) : 1 | degenere (g = 0) ? False


#### Lecture du résultat

Les deux moteurs racontent la même histoire à trois voix. La saturation a fait son office : le reste n'est pas nul, le contrat radiciel échoue. L'organe réel, laissé libre, prend la porte dégénérée ($B = A$ : hypothèses trivialement nulles, conclusion violée) — exactement la porte que la saturation ferme. Mais contraint à $B \neq A$ **et** $C \neq A$, il trouve **quand même** un témoin, avec $g \neq 0$ : un vrai triangle rectangle non isocèle. La saturation avait raison de fermer la première porte ; elle ne peut rien contre la seconde.

C'est la ligne de partage honnête du cours : **saturer purge les dégénérescences, elle ne soigne pas les énoncés faux**.


### §6 — Le prix : trois recettes, deux moteurs

Trois recettes répondent à trois questions différentes, à des coûts différents. Comme au 02 §8, on mesure :


In [16]:
# Mesures : appartenance (chaine), generateurs (elimination), contrat (radical)
t0 = time.perf_counter()
_ = sp.groebner(I_gens, *PLAN, order='lex').reduce(sp.expand(CONCL * g_nd))[1]
t1 = time.perf_counter()
saturation_par_elimination(I_gens, g_nd, PLAN)
t2 = time.perf_counter()
zp, wp = sp.symbols('_zp _wp')
_ = sp.groebner(I_gens + [zp * g_nd - 1, wp * CONCL - 1], *PLAN, zp, wp, order='lex')
t3 = time.perf_counter()

print("recette 1 : appartenance f*g^n in I   (6 var) : %.3f s" % (t1 - t0))
print("recette 2 : generateurs de I:g^infini (7 var) : %.3f s" % (t2 - t1))
print("recette 3 : contrat radiciel           (8 var) : %.3f s" % (t3 - t2))


recette 1 : appartenance f*g^n in I   (6 var) : 0.002 s
recette 2 : generateurs de I:g^infini (7 var) : 0.002 s
recette 3 : contrat radiciel           (8 var) : 0.003 s


| Recette | Question à laquelle elle répond | Coût |
|---|---|---|
| chaîne $f g^n \in I$ | « cette conclusion précise est-elle prouvée ? » | le moins cher — un test, pas un objet |
| élimination $\langle I, 1-tg\rangle \cap k[x]$ | « quel est l'idéal saturé ? » | une variable de plus — mais on garde l'objet |
| radical $\langle 1\rangle$ | « la conclusion s'annule-t-elle géométriquement ? » | deux variables de plus, et mesure la plus lente des trois |

La troisième ligne mérite une phrase, parce que la tentation est de lire son coût comme un coût de **question**. Ce n'est pas ce que les nombres disent : le contrat radiciel est le plus **lent** des trois à trancher (c'est la mesure des trois `perf_counter` ci-dessus), tout en posant la question la plus **faible** — il ne demande pas *quelle* puissance de $f$ tombe dans $I$, seulement s'il en existe une. Faible n'est pas bon marché : la faiblesse de l'énoncé ne réduit pas le coût du calcul qui le tranche.



### §7 — Le contrat de la série, relu d'un seul endroit

Toutes les méthodes de la série disent maintenant la même chose en des forces différentes :

| Position | Critère | Traduit en saturation | Portée |
|---|---|---|---|
| [02](Geometry-02-From-Equation-To-Proof.ipynb) §7 | reste nul dans $\langle I, zg-1\rangle$ | $f \in I : g^\infty$ | idéal saturé, sur $\mathbb{C}$ |
| [03](Geometry-03-Wu-Method-Python.ipynb) §5 | pseudo-reste nul (chaîne) | $f\,J^k \in I$, donc $f \in I:J^\infty$ | idéal saturé, par morceaux (chaîne) |
| [03](Geometry-03-Wu-Method-Python.ipynb) §7 | $\langle H, zJ{-}1, wg{-}1\rangle = \langle 1\rangle$ | $g \in \sqrt{I:J^\infty}$ | géométrique, sur $\mathbb{C}$ |
| [03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) | scission de Ritt | décomposition du lieu, saturation par composante | les bords où l'énoncé est muet |
| [05](Geometry-05-Pont-Formel-Python.ipynb) | preuve Lean du lac | l'hypothèse $B \neq A$ y est explicite dans l'énoncé | noyau, toutes figures concrètes |

Une seule cellule pour dérouler le fil rouge devant cette table :


In [17]:
# Le fil rouge, trois verdicts d'un coup
v_chaine = sp.groebner(I_gens, *PLAN, order='lex').reduce(sp.expand(CONCL * g_nd))[1] == 0
v_ideal = sp.groebner(sat_gens, *PLAN, order='lex').reduce(CONCL)[1] == 0
G_rad = sp.groebner(I_gens + [zp * g_nd - 1, wp * CONCL - 1], *PLAN, zp, wp, order='lex')
v_radical = (len(G_rad.polys) == 1 and sp.expand(G_rad.polys[0].as_expr()) == 1)
print("chaine (Wu/02)     : CONCL*g in I          :", v_chaine)
print("ideal sature (02b) : CONCL in I:g^infini   :", v_ideal)
print("contrat (03 §7)    : vanishing non degenere :", v_radical)
print("coherence :", v_chaine and v_ideal and v_radical)


chaine (Wu/02)     : CONCL*g in I          : True
ideal sature (02b) : CONCL in I:g^infini   : True
contrat (03 §7)    : vanishing non degenere : True
coherence : True


### §8 — Exercice 1 : saturations partielles, ou le théorème a plus d'une porte

Le 02 §11 laissait la question ouverte : que donnent les saturations **partielles** par $x_B$ seul, puis $y_B$ seul ? Montrer que chacune suffit à contenir la conclusion, puis expliquer pourquoi le 03 préfère saturer par le **produit** des non-dégénérescences.


In [18]:
# Exercice 1 : CONCL appartient-il a I : x_B^infini ? a I : y_B^infini ?
# Etape 1 : calculer les deux ideaux satures partiels par elimination
# Etape 2 : reduire CONCL par chacun
# Etape 3 : pourquoi le produit des deux composantes (B != A) est-il le geste general ?
reste_xB = None  # TODO etudiant : reste de CONCL par sat(I, x_B)
reste_yB = None  # TODO etudiant : reste de CONCL par sat(I, y_B)
print("Exercice 1 : deux saturations partielles")


Exercice 1 : deux saturations partielles


In [19]:
# Verification de l'exercice 1
# La reference ne s'affiche qu'une fois VOTRE reponse fournie : avant, elle donnerait
# la solution. Chaque branche est donc gardee par le `is not None` de sa variable.
sat_xB = saturation_par_elimination(I_gens, xB, PLAN)
sat_yB = saturation_par_elimination(I_gens, yB, PLAN)
ref_xB = sp.groebner(sat_xB, *PLAN, order='lex').reduce(CONCL)[1]
ref_yB = sp.groebner(sat_yB, *PLAN, order='lex').reduce(CONCL)[1]
if reste_xB is not None:
    print("reference : reste de CONCL par sat(I, x_B) :", ref_xB, "| nul ?", ref_xB == 0)
    print("votre reste x_B :", reste_xB, "| conforme :", bool(reste_xB) == (ref_xB == 0))
if reste_yB is not None:
    print("reference : reste de CONCL par sat(I, y_B) :", ref_yB, "| nul ?", ref_yB == 0)
    print("votre reste y_B :", reste_yB, "| conforme :", bool(reste_yB) == (ref_yB == 0))
if reste_xB is None and reste_yB is None:
    print("Exercice 1 a completer : renseignez reste_xB et/ou reste_yB ci-dessus.")



Exercice 1 a completer : renseignez reste_xB et/ou reste_yB ci-dessus.


#### Lecture de l'exercice 1

Chaque saturation partielle suffit : sur le lieu $x_B \neq 0$, les hypothèses forcent $C$ et $D$ sur une même droite issue de $A$ (et symétriquement sur $y_B \neq 0$). Les deux lieux couvrent $V(I) \setminus \{B = A\}$ — le théorème tient sur l'union. Mais un seul énoncé par **produit** $x_B \cdot y_B \neq 0$ serait faux (il exclurait trop) : c'est $g_{nd} = x_B^2 + y_B^2$ qui partitionne exactement, et c'est ce genre de partition que le produit des initiales de Wu réalise systématiquement (03 §6).


### §9 — Exercice 2 : un idéal que la saturation transforme vraiment

Sur le fil rouge, l'idéal saturé ne diffère de $I$ que par la conclusion. Ce n'est pas toujours aussi discret. Prenons $I = \langle x y,\ x z\rangle$ et $g = y$ :

1. calculer $I : y^\infty$ par élimination ;
2. vérifier que l'inclusion $I \subsetneq I : y^\infty$ est **stricte** ;
3. lire géométriquement : $V(I)$ est l'union de deux lieux — lequel survive-t-il après avoir retiré $V(y)$ ?


In [20]:
# Exercice 2 : saturation stricte
# Etape 1 : I:y^infini par elimination (3 variables x, y, z puis variable auxiliaire)
# Etape 2 : trouver un polynome dans I:y^infini qui n'est pas dans I
# Etape 3 : decrire V(I) et ce que la saturation par y en retire
sat_ex2 = None   # TODO etudiant : generateurs de <x*y, x*z> : y^infini
temoin_ex2 = None  # TODO etudiant : un polynome de I:y^infini hors de I
print("Exercice 2 : la saturation stricte")


Exercice 2 : la saturation stricte


In [21]:
# Verification de l'exercice 2
# Meme garde que pour l'exercice 1 : les deux impressions de reference sont
# conditionnees a une tentative, sans quoi la cellule livre la reponse.
xa, ya, za = sp.symbols('x_a y_a z_a')
I_ex2 = [xa * ya, xa * za]
sat_ref = saturation_par_elimination(I_ex2, ya, (xa, ya, za))
G_I2 = sp.groebner(I_ex2, xa, ya, za, order='lex')
hors = [p for p in sat_ref if G_I2.reduce(p)[1] != 0]
if sat_ex2 is not None or temoin_ex2 is not None:
    print("reference : I : y^infini =", sat_ref)
    print("generateurs satures hors de I :", hors)
else:
    print("Exercice 2 a completer : renseignez sat_ex2 et/ou temoin_ex2 ci-dessus.")
if sat_ex2 is not None:
    G_e2 = sp.groebner(list(sat_ex2), xa, ya, za, order='lex')
    ok = all(G_e2.reduce(p)[1] == 0 for p in sat_ref) and all(
        sp.groebner(sat_ref, xa, ya, za, order='lex').reduce(p)[1] == 0 for p in sat_ex2)
    print("votre ideal coincide avec la reference :", ok)
if temoin_ex2 is not None:
    print("votre temoin est hors de I :", G_I2.reduce(sp.expand(temoin_ex2))[1] != 0)



Exercice 2 a completer : renseignez sat_ex2 et/ou temoin_ex2 ci-dessus.


#### Lecture de l'exercice 2

$V(x y, x z)$ est l'union du plan $\{x = 0\}$ et de la droite $\{y = z = 0\}$. Retirer $V(y)$ (le plan $y = 0$) coupe la droite en un point mais ne touche pas au plan $\{x=0\}$ hors de $y=0$ ; la clôture du reste est le plan entier $\{x = 0\}$ — et $I : y^\infty = \langle x\rangle$. La saturation a **supprimé une composante** de l'idéal : sur le fil rouge elle ajoutait la conclusion, ici elle retire de l'hypothèse ce qui ne vivait que sur la partie dégénérée.


### §10 — Exercice 3 : votre premier certificat de Nullstellensatz

Avec $J = \langle x^2,\ x y\rangle$ et $f = x$ : $f$ s'annule sur $V(J)$.

1. trouver le plus petit $m$ tel que $f^m \in J$ ;
2. construire une identité $1 = b\,(1 - t x) + a\,x^2$ (celle du §4 fonctionne telle quelle), la vérifier par développement, puis substituer $t = 1/x$ et nettoyer par $x^m$ — l'identité finale doit être un certificat de $x^m \in J$.


In [22]:
# Exercice 3 : le certificat complet
# Etape 1 : minimalite de m (tester m = 1, 2, 3 par reduce contre GB(J))
# Etape 2 : reduce(1) sur <J, 1 - t*x>, substitution t = 1/x, multiplication par x^m
# Etape 3 : verifier que la somme obtenue redonne x^m comme combination de x^2 et x*y
m_ex3 = None       # TODO etudiant : plus petit m tel que x^m in <x^2, x*y>
cofacteurs_ex3 = None  # TODO etudiant : les cofacteurs (devant x^2 et x*y) de l'identite finale
print("Exercice 3 : certificat de Nullstellensatz")


Exercice 3 : certificat de Nullstellensatz


In [23]:
# Verification de l'exercice 3
xb_, yb_ = sp.symbols('x_b y_b')
J_ex3 = [xb_ ** 2, xb_ * yb_]
G_ex3 = sp.groebner(J_ex3, xb_, yb_, order='lex')
p_min = next(p for p in (1, 2, 3) if G_ex3.reduce(xb_ ** p)[1] == 0)

# derivation : 1 = (1+t*x)*(1-t*x) + t^2*x^2, substitution t = 1/x, nettoyage
m_ref = max(sp.degree(1 + t * xb_, t), sp.degree(t ** 2, t))
coeff_a = sp.expand((t ** 2).subs(t, 1 / xb_) * xb_ ** m_ref)
cert = sp.expand(coeff_a * xb_ ** 2)

# Les deux valeurs de reference ne s'impriment qu'apres une tentative (exercices 1 et 2
# suivent la meme regle) : sinon la cellule donne m et le certificat sans rien demander.
if m_ex3 is not None or cofacteurs_ex3 is not None:
    print(f"reference : exposant minimal m = {p_min} | x^{p_min - 1} in J :",
          G_ex3.reduce(xb_ ** (p_min - 1))[1] == 0)
    print(f"reference : derivation -> x^{m_ref} =", cert,
          "| certificat :", sp.expand(cert - xb_ ** m_ref) == 0)
else:
    print("Exercice 3 a completer : renseignez m_ex3 et/ou cofacteurs_ex3 ci-dessus.")
if m_ex3 is not None:
    print("votre m :", m_ex3, "| conforme :", m_ex3 == p_min)
if cofacteurs_ex3 is not None:
    exp = xb_ ** m_ex3 if m_ex3 is not None else xb_ ** m_ref
    ident = sp.expand(sum(c * g for c, g in zip(cofacteurs_ex3, J_ex3)))
    print("votre identite somme c_i*g_i =", ident, "| exacte :", sp.expand(ident - exp) == 0)



Exercice 3 a completer : renseignez m_ex3 et/ou cofacteurs_ex3 ci-dessus.


#### Lecture de l'exercice 3

$x$ n'est pas dans $J$ (reste non nul), $x^2$ y est — $m = 2$. Le certificat final n'est pas un artefact : chaque coefficient de l'identité est polynomial, et la somme reconstruite redonne exactement $x^2$ comme combinaison de $x^2$ et $xy$. Vous venez d'exécuter, sur l'exemple fixe ci-dessus ($J = \langle x^2, xy\rangle$, $f = x$), la preuve du Nullstellensatz fort — la même que celle qui garantit les contrats radiciels du 03.


## Conclusion

Ce carnet a nommé et calculé l'objet que la série manipulait depuis le 02 :

| Brique | Rôle | Où |
|---|---|---|
| chaîne $I : g^n$ | le test d'appartenance — ce que Wu établit | §2 |
| élimination $\langle I, 1-tg\rangle \cap k[x]$ | les **générateurs** de l'idéal saturé | §2 |
| théorème de Ritt–Rabinowitsch | l'astuce du 02/03 EST l'algorithme de saturation | §3 |
| dérivation faible $\to$ fort | Nullstellensatz exécuté, cofacteurs à l'appui | §4 |
| contrat radiciel | ce que « prouve modulo dégénérescences » garantit sur $\mathbb{C}$ | §4, §7 |
| témoin non dégénéré | la saturation ne soigne pas les énoncés faux — l'organe Z3 trouve le témoin | §5 |

**Limites honnêtes.** (1) Le contrat radiciel vit sur $\mathbb{C}$ ; la subtilité réel vs complexe reste celle du 02 §12 et de l'organe `real_geometry_bridge`. (2) Quand la conclusion est un **quotient** et non un polynôme, la scission de Ritt du 03b décompose ce que la saturation globalise — les deux lectures se complètent. (3) Le lac Lean du 05 prouve l'énoncé avec l'hypothèse $B \neq A$ ** explicite** : la preuve au noyau ne sature rien, elle hypothèque — c'est une force, pas un doublon.


## Coût et reproductibilité

- CPU uniquement (sympy, Z3 via l'organe de la série) ; aucune API, aucun GPU, aucun réseau ; déterministe (aucun tirage aléatoire).
- Détail chiffré dans le bloc `metadata.cost` du carnet.


## À retenir

- **Deux routes, un seul verdict** : l'élimination (qui calcule l'idéal saturé) et l'astuce `z \cdot g - 1` du §7 du 02 convergent — `deux routes, un seul verdict : True`.
- **L'idéal saturé est un idéal du plan** : l'élimination de la variable auxiliaire rend **trois** générateurs sans variable auxiliaire — les deux hypothèses et la conclusion elle-même — et `I : g^1 ⊂ I : g^2 ⊂ I : g^3` est vérifié, avec `I : g^3 = I : g^∞` (mutuellement).
- **Le Nullstellensatz faible se dérive en fort, par machine** : sur `J = ⟨x^2, xy⟩`, la base de `⟨x^2, 1 - t·x⟩` est `[Poly(1, x, t, domain='ZZ')]`, l'ideal est `<1>`, et le certificat `1 = (t·x + 1)(1 - t·x) + t^2 · x^2` se vérifie avec un écart nul.
- **L'exposant minimal se mesure** : `x^1` n'est pas dans `J`, `x^2` y est — l'exposant minimal est `2`, et la puissance est donc **nécessaire**, pas choisie.
- **La saturation ne soigne pas les énoncés faux** : le reste de `Ciso` par l'idéal saturé reste non nul, et Z3 produit un témoin réel **non dégénéré** (`x_B2 = -1`, `y_C2 = -2`) — un énoncé faux reste faux après saturation.
- **Les trois portes concordent** : `chaîne (Wu/02) : CONCL*g in I : True`, `idéal saturé (02b) : CONCL in I:g^∞ : True`, `contrat (03 §7) : vanishing non dégénéré : True` — `cohérence : True`.


## Vérifiez votre compréhension

1. Les deux routes de saturation — élimination, et `z \cdot g - 1` — rendent le même verdict. Que gagne-t-on à faire coexister deux routes indépendantes ?
2. L'idéal saturé est rendu par trois générateurs **sans variable auxiliaire**, dont la conclusion elle-même. Comment lit-on ce résultat : « la conclusion est dans l'idéal », ou « la conclusion est un générateur » ?
3. Sur `J = ⟨x^2, xy⟩`, l'exposant minimal est `2` : `x^1` n'est pas dans `J`, `x^2` y est. Quel est le rôle exact de cet exposant dans le Nullstellensatz fort ?
4. La saturation ne change pas le sort d'un énoncé faux : `Ciso` garde un reste non nul, et Z3 trouve un témoin **non dégénéré**. Que distingue ce témoin de celui que le solveur rend sans contrainte ?

<details>
<summary>Réponses</summary>

1. L'**indépendance** des routes : chacune peut échouer pour une raison qui n'a rien à voir avec la vérité de l'énoncé ; deux routes qui convergent vers le même verdict rendent la conclusion robuste à l'échec de l'une d'elles. Ici l'une est un calcul (élimination), l'autre l'astuce algébrique du Nullstellensatz : elles ne partagent pas la même implémentation.
2. Les deux lectures sont vraies, et la seconde est la plus forte : la conclusion n'est pas seulement **dans** l'idéal saturé, elle en est un **générateur** — l'appartenance est exhibée, pas seulement constatée. C'est ce qui permet de relire le théorème sans refaire l'élimination.
3. Il fixe la **puissance** à laquelle le polynôme `f` doit être multiplié pour entrer dans l'ideal : `f^m ∈ J`. Le Nullstellensatz fort dit que `f ∈ rad(J)`, soit `f^m ∈ J` pour un certain `m` ; la mesure montre que `m = 2` est **nécessaire** ici — `x` seul ne suffit pas — et le certificat `1 = (t·x + 1)(1 - t·x) + t^2 · x^2` le rend explicite.
4. Le témoin rendu sans contrainte est **dégénéré** (`x_A = 0`, `y_A = 0`, `x_B2 = 0`, `y_C2 = -1`, soit `g = 0`) : il annule la condition même que la saturation imposait. Celui de la §5 est obtenu **après** saturation, donc avec `g ≠ 0` forcé — `x_B2 = -1`, `y_C2 = -2` ne sont pas un point de dégénérescence. La saturation ne « soigne » rien : elle change la **classe** des témoins autorisés, et l'énoncé faux reste faux.

</details>


## Pour aller plus loin

- J. L. Rabinowitsch, *Zum Hilbertschen Nullstellensatz* (1929) — l'astuce d'une variable, dans sa lettre originale.
- D. Cox, J. Little, D. O'Shea, *Ideals, Varieties, and Algorithms* (4e ed.), chap. 2 §2 (Nullstellensatz) et chap. 4 §4 (saturation, élimination) — la dérivation exécutée en §4 en est le Théorème 2.2.10 poussé en machine.
- D. Eisenbud, *Commutative Algebra with a View Toward Algebraic Geometry*, chap. 4 — saturation et décomposition primaire.
- S.-C. Chou, *Mechanical Geometry Theorem Proving* (1988) — la lecture « non-dégénérescences » de la méthode de Wu.

Série : [Geometry](README.md) — la preuve automatique en géométrie, de la figure à la preuve. Suivant logique : [03b](Geometry-03b-Ritt-Decomposition-Python.ipynb) pour la décomposition, [05](Geometry-05-Pont-Formel-Python.ipynb) pour le pont au noyau.
